In [1]:
import sys
import pandas as pd

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("Setup is ready!")


Python version: 3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]
Pandas version: 3.0.5
Setup is ready!


In [2]:
import pandas as pd

# Reference dataset: approved product codes
reference_data = pd.DataFrame({
    "product_code": ["P101", "P102", "P103"],
    "status": ["active", "active", "deprecated"],
    "valid_from": ["2026-01-01", "2026-01-01", "2026-01-01"],
    "valid_to": [None, None, "2026-06-30"]
})

# Incoming records to validate
incoming_data = pd.DataFrame({
    "record_id": [1, 2, 3, 4],
    "product_code": ["P101", "P102", "P999", "P103"],
    "event_date": ["2026-03-15", "2026-04-10", "2026-03-20", "2026-08-01"]
})

print("REFERENCE DATA")
display(reference_data)

print("INCOMING DATA")
display(incoming_data)

REFERENCE DATA


,product_code,status,valid_from,valid_to
0,P101,active,2026-01-01,NaN
1,P102,active,2026-01-01,NaN
2,P103,deprecated,2026-01-01,2026-06-30


INCOMING DATA


,record_id,product_code,event_date
0,1,P101,2026-03-15
1,2,P102,2026-04-10
2,3,P999,2026-03-20
3,4,P103,2026-08-01


In [3]:
# Convert date columns to real dates
reference_data["valid_from"] = pd.to_datetime(reference_data["valid_from"])
reference_data["valid_to"] = pd.to_datetime(reference_data["valid_to"])
incoming_data["event_date"] = pd.to_datetime(incoming_data["event_date"])

# Match incoming records with reference codes
results = incoming_data.merge(
    reference_data,
    on="product_code",
    how="left"
)

# Apply validation rules
def validate_record(row):
    if pd.isna(row["status"]):
        return "UNKNOWN_CODE"

    if row["event_date"] < row["valid_from"]:
        return "NOT_YET_VALID"

    if pd.notna(row["valid_to"]) and row["event_date"] > row["valid_to"]:
        return "EXPIRED_CODE"

    if row["status"] == "deprecated":
        return "DEPRECATED_CODE"

    return "VALID"

results["validation_status"] = results.apply(validate_record, axis=1)

display(results[
    ["record_id", "product_code", "event_date", "validation_status"]
])

,record_id,product_code,event_date,validation_status
0,1,P101,2026-03-15,VALID
1,2,P102,2026-04-10,VALID
2,3,P999,2026-03-20,UNKNOWN_CODE
3,4,P103,2026-08-01,EXPIRED_CODE


In [4]:
# Separate valid and rejected records
valid_records = results[
    results["validation_status"] == "VALID"
].copy()

rejected_records = results[
    results["validation_status"] != "VALID"
].copy()

# Display summary
print("Total input records:", len(incoming_data))
print("Valid records:", len(valid_records))
print("Rejected records:", len(rejected_records))

print("\nREJECTED RECORDS")
display(
    rejected_records[
        ["record_id", "product_code", "event_date", "validation_status"]
    ]
)

Total input records: 4
Valid records: 2
Rejected records: 2

REJECTED RECORDS


,record_id,product_code,event_date,validation_status
2,3,P999,2026-03-20,UNKNOWN_CODE
3,4,P103,2026-08-01,EXPIRED_CODE


In [5]:
# Save accepted and rejected records
valid_records.to_csv("valid_records.csv", index=False)
rejected_records.to_csv("rejected_records.csv", index=False)

# Save a simple run summary
run_summary = pd.DataFrame([{
    "total_input": len(incoming_data),
    "valid_records": len(valid_records),
    "rejected_records": len(rejected_records)
}])

run_summary.to_csv("run_summary.csv", index=False)

print("Output files saved successfully!")
print("1. valid_records.csv")
print("2. rejected_records.csv")
print("3. run_summary.csv")

Output files saved successfully!
1. valid_records.csv
2. rejected_records.csv
3. run_summary.csv


In [6]:
# Create versioned reference data
versioned_reference = pd.DataFrame({
    "product_code": ["P101", "P102", "P103", "P101", "P102"],
    "reference_version": [1, 1, 1, 2, 2],
    "status": [
        "active", "active", "deprecated",
        "active", "deprecated"
    ],
    "valid_from": [
        "2026-01-01", "2026-01-01", "2026-01-01",
        "2026-07-01", "2026-07-01"
    ],
    "valid_to": [
        None, None, "2026-06-30",
        None, None
    ]
})

versioned_reference["valid_from"] = pd.to_datetime(
    versioned_reference["valid_from"]
)
versioned_reference["valid_to"] = pd.to_datetime(
    versioned_reference["valid_to"]
)

display(versioned_reference)

,product_code,reference_version,status,valid_from,valid_to
0,P101,1,active,2026-01-01,NaT
1,P102,1,active,2026-01-01,NaT
2,P103,1,deprecated,2026-01-01,2026-06-30
3,P101,2,active,2026-07-01,NaT
4,P102,2,deprecated,2026-07-01,NaT


In [7]:
# Compare product status across reference versions
version_comparison = versioned_reference[
    ["product_code", "reference_version", "status"]
].sort_values(["product_code", "reference_version"])

display(version_comparison)

# Check the expected version-specific statuses
p102_versions = version_comparison[
    version_comparison["product_code"] == "P102"
]

assert list(p102_versions["status"]) == ["active", "deprecated"]

print("Version comparison passed!")

,product_code,reference_version,status
0,P101,1,active
3,P101,2,active
1,P102,1,active
4,P102,2,deprecated
2,P103,1,deprecated


Version comparison passed!


In [8]:
# Sample incoming records with explicit reference versions
versioned_incoming = pd.DataFrame({
    "record_id": [101, 102, 103, 104],
    "product_code": ["P102", "P102", "P999", "P103"],
    "event_date": [
        "2026-05-15",
        "2026-08-15",
        "2026-08-15",
        "2026-05-15"
    ],
    "reference_version": [1, 2, 2, 1]
})

versioned_incoming["event_date"] = pd.to_datetime(
    versioned_incoming["event_date"]
)

display(versioned_incoming)

,record_id,product_code,event_date,reference_version
0,101,P102,2026-05-15,1
1,102,P102,2026-08-15,2
2,103,P999,2026-08-15,2
3,104,P103,2026-05-15,1


In [9]:
def validate_versioned_record(row):
    # Select the matching code and reference version
    matches = versioned_reference[
        (versioned_reference["product_code"] == row["product_code"]) &
        (versioned_reference["reference_version"] == row["reference_version"])
    ]

    if matches.empty:
        return "UNKNOWN_CODE"

    ref = matches.iloc[0]

    if row["event_date"] < ref["valid_from"]:
        return "NOT_YET_VALID"

    if pd.notna(ref["valid_to"]) and row["event_date"] > ref["valid_to"]:
        return "EXPIRED_CODE"

    if ref["status"] == "deprecated":
        return "DEPRECATED_CODE"

    return "VALID"


versioned_incoming["validation_status"] = versioned_incoming.apply(
    validate_versioned_record,
    axis=1
)

display(versioned_incoming)

,record_id,product_code,event_date,reference_version,validation_status
0,101,P102,2026-05-15,1,VALID
1,102,P102,2026-08-15,2,DEPRECATED_CODE
2,103,P999,2026-08-15,2,UNKNOWN_CODE
3,104,P103,2026-05-15,1,DEPRECATED_CODE


In [10]:
# Expected results for our four test records
expected_statuses = [
    "VALID",
    "DEPRECATED_CODE",
    "UNKNOWN_CODE",
    "DEPRECATED_CODE"
]

actual_statuses = versioned_incoming["validation_status"].tolist()

assert actual_statuses == expected_statuses, (
    f"Expected {expected_statuses}, got {actual_statuses}"
)

print("All 4 version-aware validation tests passed!")

All 4 version-aware validation tests passed!


In [11]:
# Create a test batch containing a duplicate record
duplicate_test_data = pd.DataFrame({
    "record_id": [201, 202, 202, 203],
    "product_code": ["P101", "P102", "P102", "P999"],
    "event_date": [
        "2026-05-01",
        "2026-05-10",
        "2026-05-10",
        "2026-05-15"
    ]
})

# Identify all occurrences of duplicate record IDs
duplicate_test_data["is_duplicate"] = duplicate_test_data.duplicated(
    subset=["record_id"],
    keep=False
)

display(duplicate_test_data)

,record_id,product_code,event_date,is_duplicate
0,201,P101,2026-05-01,False
1,202,P102,2026-05-10,True
2,202,P102,2026-05-10,True
3,203,P999,2026-05-15,False


In [12]:
# Separate duplicate and unique records
duplicate_records = duplicate_test_data[
    duplicate_test_data["is_duplicate"]
].copy()

unique_records = duplicate_test_data[
    ~duplicate_test_data["is_duplicate"]
].copy()

print("Total input records:", len(duplicate_test_data))
print("Unique records:", len(unique_records))
print("Duplicate records:", len(duplicate_records))

print("\nDUPLICATE RECORDS")
display(duplicate_records)

Total input records: 4
Unique records: 2
Duplicate records: 2

DUPLICATE RECORDS


,record_id,product_code,event_date,is_duplicate
1,202,P102,2026-05-10,True
2,202,P102,2026-05-10,True


In [13]:
# Add an explicit rejection reason
duplicate_records["rejection_reason"] = "DUPLICATE_RECORD_ID"

# Add a processing outcome
duplicate_records["processing_status"] = "REJECTED"

display(duplicate_records)

,record_id,product_code,event_date,is_duplicate,rejection_reason,processing_status
1,202,P102,2026-05-10,True,DUPLICATE_RECORD_ID,REJECTED
2,202,P102,2026-05-10,True,DUPLICATE_RECORD_ID,REJECTED


In [14]:
# Save duplicate records for review
duplicate_records.to_csv(
    "duplicate_records.csv",
    index=False
)

print("Duplicate report saved successfully!")
print("File: duplicate_records.csv")
print("Rows saved:", len(duplicate_records))

Duplicate report saved successfully!
File: duplicate_records.csv
Rows saved: 2


In [15]:
from datetime import datetime, timezone
import uuid

# Generate a unique ID for this pipeline run
run_id = str(uuid.uuid4())

# Record the run start time in UTC
run_start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Run start time (UTC):", run_start_time)

Run ID: ce684c95-4dcf-4bc9-8437-17174fae3bb9
Run start time (UTC): 2026-10-31T07:12:16.249015+00:00


In [16]:
from datetime import datetime, timezone

run_end_time = datetime.now(timezone.utc).isoformat()

run_log = pd.DataFrame([{
    "run_id": run_id,
    "start_time_utc": run_start_time,
    "end_time_utc": run_end_time,
    "input_count": len(versioned_incoming),
    "accepted_count": (
        versioned_incoming["validation_status"] == "VALID"
    ).sum(),
    "rejected_count": (
        versioned_incoming["validation_status"] != "VALID"
    ).sum(),
    "run_status": "SUCCESS"
}])

display(run_log)

run_log.to_csv("run_log.csv", index=False)

print("Run log saved successfully!")

,run_id,start_time_utc,end_time_utc,input_count,accepted_count,rejected_count,run_status
0,ce684c95-4dcf-4bc9-8437-17174fae3bb9,2026-10-31T07:12:16.249015+00:00,2026-10-31T07:12:16.714794+00:00,4,1,3,SUCCESS


Run log saved successfully!


In [17]:
# Reconcile input, accepted, and rejected records
total_input = len(versioned_incoming)

accepted_count = (
    versioned_incoming["validation_status"] == "VALID"
).sum()

rejected_count = (
    versioned_incoming["validation_status"] != "VALID"
).sum()

print("Input records:", total_input)
print("Accepted records:", accepted_count)
print("Rejected records:", rejected_count)
print("Accepted + rejected:", accepted_count + rejected_count)

assert total_input == accepted_count + rejected_count

print("\nRECONCILIATION PASSED!")

Input records: 4
Accepted records: 1
Rejected records: 3
Accepted + rejected: 4

RECONCILIATION PASSED!


In [18]:
# Create test records with a missing product code
missing_test_data = pd.DataFrame({
    "record_id": [301, 302, 303],
    "product_code": ["P101", None, "P102"],
    "event_date": ["2026-05-01", "2026-05-02", None]
})

# Identify missing required fields
required_columns = ["record_id", "product_code", "event_date"]

missing_test_data["missing_fields"] = missing_test_data[
    required_columns
].isna().apply(
    lambda row: ", ".join(row.index[row].tolist()),
    axis=1
)

missing_test_data["validation_status"] = missing_test_data[
    "missing_fields"
].apply(
    lambda fields: "REJECTED_MISSING_FIELD" if fields else "READY_FOR_VALIDATION"
)

display(missing_test_data)

,record_id,product_code,event_date,missing_fields,validation_status
0,301,P101,2026-05-01,,READY_FOR_VALIDATION
1,302,NaN,2026-05-02,product_code,REJECTED_MISSING_FIELD
2,303,P102,NaN,event_date,REJECTED_MISSING_FIELD


In [19]:
# Add explicit rejection reasons
missing_test_data["rejection_reason"] = missing_test_data[
    "missing_fields"
].apply(
    lambda fields: (
        "MISSING_REQUIRED_FIELD: " + fields
        if fields
        else None
    )
)

display(missing_test_data[
    ["record_id", "product_code", "event_date",
     "validation_status", "rejection_reason"]
])

,record_id,product_code,event_date,validation_status,rejection_reason
0,301,P101,2026-05-01,READY_FOR_VALIDATION,NaN
1,302,NaN,2026-05-02,REJECTED_MISSING_FIELD,MISSING_REQUIRED_FIELD: product_code
2,303,P102,NaN,REJECTED_MISSING_FIELD,MISSING_REQUIRED_FIELD: event_date


In [20]:
# Create a reference dataset with a duplicate key
reference_key_test = pd.DataFrame({
    "product_code": ["P101", "P102", "P102"],
    "reference_version": [1, 1, 1],
    "status": ["active", "active", "deprecated"]
})

# A code should appear only once per reference version
reference_key_test["is_duplicate_key"] = (
    reference_key_test.duplicated(
        subset=["product_code", "reference_version"],
        keep=False
    )
)

display(reference_key_test)

duplicate_keys = reference_key_test[
    reference_key_test["is_duplicate_key"]
]

print("Duplicate reference-key rows:", len(duplicate_keys))

,product_code,reference_version,status,is_duplicate_key
0,P101,1,active,False
1,P102,1,active,True
2,P102,1,deprecated,True


Duplicate reference-key rows: 2


In [21]:
# Check for duplicate reference keys
duplicate_key_count = reference_key_test.duplicated(
    subset=["product_code", "reference_version"]
).sum()

if duplicate_key_count > 0:
    print("REFERENCE DATA ERROR")
    print("Duplicate keys found:", duplicate_key_count)
    print("Affected records:")
    display(
        reference_key_test[
            reference_key_test.duplicated(
                subset=["product_code", "reference_version"],
                keep=False
            )
        ]
    )
else:
    print("Reference keys are unique.")

REFERENCE DATA ERROR
Duplicate keys found: 1
Affected records:


,product_code,reference_version,status,is_duplicate_key
1,P102,1,active,True
2,P102,1,deprecated,True


In [22]:
# Create an empty input dataset with the expected schema
empty_input = pd.DataFrame(
    columns=["record_id", "product_code", "event_date"]
)

# Check and report the empty batch
if empty_input.empty:
    print("EMPTY INPUT BATCH")
    print("Input records: 0")
    print("Accepted records: 0")
    print("Rejected records: 0")
    print("Pipeline handled empty input safely.")
else:
    print("Input records:", len(empty_input))

EMPTY INPUT BATCH
Input records: 0
Accepted records: 0
Rejected records: 0
Pipeline handled empty input safely.


In [23]:
# Create test records with valid and invalid dates
date_test_data = pd.DataFrame({
    "record_id": [401, 402, 403],
    "product_code": ["P101", "P102", "P103"],
    "event_date": [
        "2026-05-01",
        "not-a-date",
        "2026-06-15"
    ]
})

# Parse dates safely; invalid dates become missing values
date_test_data["parsed_event_date"] = pd.to_datetime(
    date_test_data["event_date"],
    errors="coerce"
)

# Flag invalid or missing dates
date_test_data["validation_status"] = date_test_data[
    "parsed_event_date"
].isna().map({
    True: "REJECTED_INVALID_DATE",
    False: "READY_FOR_VALIDATION"
})

display(date_test_data)

,record_id,product_code,event_date,parsed_event_date,validation_status
0,401,P101,2026-05-01,2026-05-01,READY_FOR_VALIDATION
1,402,P102,not-a-date,NaT,REJECTED_INVALID_DATE
2,403,P103,2026-06-15,2026-06-15,READY_FOR_VALIDATION


In [24]:
# Extract records with invalid dates
invalid_date_records = date_test_data[
    date_test_data["validation_status"] == "REJECTED_INVALID_DATE"
].copy()

# Add an explicit rejection reason
invalid_date_records["rejection_reason"] = "INVALID_EVENT_DATE"

# Save the report
invalid_date_records.to_csv(
    "invalid_date_rejections.csv",
    index=False
)

print("Invalid-date rejection report saved!")
print("Rejected rows:", len(invalid_date_records))
display(invalid_date_records)

Invalid-date rejection report saved!
Rejected rows: 1


,record_id,product_code,event_date,parsed_event_date,validation_status,rejection_reason
1,402,P102,not-a-date,NaT,REJECTED_INVALID_DATE,INVALID_EVENT_DATE


In [25]:
# Simulate the same input being processed twice
rerun_test_data = pd.concat(
    [versioned_incoming, versioned_incoming],
    ignore_index=True
)

print("Rows before deduplication:", len(rerun_test_data))

# Keep one row per record ID for this demonstration
rerun_output = rerun_test_data.drop_duplicates(
    subset=["record_id"],
    keep="first"
).copy()

print("Rows after deduplication:", len(rerun_output))
print("Unique record IDs:", rerun_output["record_id"].nunique())

assert rerun_output["record_id"].is_unique
assert len(rerun_output) == len(versioned_incoming)

print("Idempotency demonstration passed!")

Rows before deduplication: 8
Rows after deduplication: 4
Unique record IDs: 4
Idempotency demonstration passed!


In [26]:
import time

# Measure validation runtime
start_time = time.perf_counter()

performance_results = versioned_incoming.copy()

performance_results["validation_status"] = (
    performance_results.apply(
        validate_versioned_record,
        axis=1
    )
)

elapsed_seconds = time.perf_counter() - start_time

print("Records processed:", len(performance_results))
print(f"Execution time: {elapsed_seconds:.6f} seconds")

if elapsed_seconds > 0:
    print(
        "Throughput:",
        round(len(performance_results) / elapsed_seconds, 2),
        "records/second"
    )

Records processed: 4
Execution time: 0.015983 seconds
Throughput: 250.26 records/second


In [27]:
import numpy as np

# Generate reproducible synthetic input data
rng = np.random.default_rng(42)

n_records = 100_000

large_incoming = pd.DataFrame({
    "record_id": np.arange(1, n_records + 1),
    "product_code": rng.choice(
        ["P101", "P102", "P103", "P999"],
        size=n_records
    ),
    "event_date": rng.choice(
        pd.to_datetime([
            "2026-05-15",
            "2026-06-15",
            "2026-08-15"
        ]),
        size=n_records
    ),
    "reference_version": rng.choice(
        [1, 2],
        size=n_records
    )
})

print("Generated records:", len(large_incoming))
print("Random seed: 42")
print("Unique record IDs:", large_incoming["record_id"].nunique())

display(large_incoming.head())

Generated records: 100000
Random seed: 42
Unique record IDs: 100000


,record_id,product_code,event_date,reference_version
0,1,P101,2026-08-15,2
1,2,P999,2026-08-15,2
2,3,P103,2026-06-15,2
3,4,P102,2026-06-15,2
4,5,P102,2026-05-15,2


In [31]:
print("Checking Python kernel...")

import pipeline

print("SUCCESS: Pipeline imported!")

Checking Python kernel...
SUCCESS: Pipeline imported!


In [67]:
# Verify benchmark output
assert len(benchmark_data) == 100_000
assert benchmark_data["validation_status"].notna().all()

status_counts = benchmark_data["validation_status"].value_counts()

assert status_counts.sum() == len(benchmark_data)

print("Benchmark validation passed!")
print("Total records:", len(benchmark_data))
print("Records with a status:", benchmark_data["validation_status"].notna().sum())
print("Status counts reconcile:", status_counts.sum() == len(benchmark_data))

KeyError: 'validation_status'

In [ ]:
# Save the full validation output
benchmark_data.to_csv(
    "large_validation_results.csv",
    index=False
)

# Save the status counts
status_report = (
    benchmark_data["validation_status"]
    .value_counts()
    .rename_axis("validation_status")
    .reset_index(name="record_count")
)

status_report.to_csv("validation_status_report.csv", index=False)

print("Benchmark results saved successfully!")
print("Output rows:", len(benchmark_data))
print("Files created:")
print("- large_validation_results.csv")
print("- validation_status_report.csv")

In [68]:
import os
import pandas as pd

# Check that both output files exist
files_to_check = [
    "large_validation_results.csv",
    "validation_status_report.csv"
]

for filename in files_to_check:
    print(filename, "exists:", os.path.exists(filename))

# Read the saved validation results
saved_results = pd.read_csv("large_validation_results.csv")

print("\nSaved records:", len(saved_results))
print(
    "Validation statuses missing:",
    saved_results["validation_status"].isna().sum()
)

assert len(saved_results) == 100_000
assert saved_results["validation_status"].notna().all()

print("\nSaved output verification passed!")

large_validation_results.csv exists: True
validation_status_report.csv exists: True

Saved records: 100000
Validation statuses missing: 0

Saved output verification passed!


In [ ]:
data_dictionary = pd.DataFrame([
    {
        "column_name": "record_id",
        "data_type": "integer",
        "description": "Unique identifier for each incoming record"
    },
    {
        "column_name": "product_code",
        "data_type": "string",
        "description": "Code being validated against reference data"
    },
    {
        "column_name": "event_date",
        "data_type": "date",
        "description": "Date the incoming record applies to"
    },
    {
        "column_name": "reference_version",
        "data_type": "integer",
        "description": "Version of the reference dataset used"
    },
    {
        "column_name": "validation_status",
        "data_type": "string",
        "description": "Validation outcome for the incoming record"
    }
])

display(data_dictionary)

data_dictionary.to_csv("data_dictionary.csv", index=False)

print("\nData dictionary saved successfully!")

In [69]:
# Save the versioned reference dataset
versioned_reference.to_csv(
    "versioned_reference.csv",
    index=False
)

print("Reference dataset saved successfully!")
print("Rows saved:", len(versioned_reference))
print("Reference versions:")
print(
    sorted(
        versioned_reference["reference_version"].unique()
    )
)

Reference dataset saved successfully!
Rows saved: 5
Reference versions:
[np.int64(1), np.int64(2)]


In [70]:
# Reload the saved reference dataset
saved_reference = pd.read_csv("versioned_reference.csv")

print("Saved reference rows:", len(saved_reference))
print(
    "Saved versions:",
    sorted(saved_reference["reference_version"].unique())
)

display(saved_reference)

assert len(saved_reference) == 5
assert set(saved_reference["reference_version"]) == {1, 2}

print("Reference dataset verification passed!")

Saved reference rows: 5
Saved versions: [np.int64(1), np.int64(2)]


,product_code,reference_version,status,valid_from,valid_to
0,P101,1,active,2026-01-01,NaN
1,P102,1,active,2026-01-01,NaN
2,P103,1,deprecated,2026-01-01,2026-06-30
3,P101,2,active,2026-07-01,NaN
4,P102,2,deprecated,2026-07-01,NaN


Reference dataset verification passed!


In [71]:
from validator import validate_record

print("Validator module imported successfully!")

Validator module imported successfully!


In [72]:
# Ensure reference dates are parsed correctly
saved_reference["valid_from"] = pd.to_datetime(
    saved_reference["valid_from"]
)
saved_reference["valid_to"] = pd.to_datetime(
    saved_reference["valid_to"]
)

# Test a valid product code
test_record = {
    "record_id": 501,
    "product_code": "P101",
    "event_date": "2026-05-15",
    "reference_version": 1
}

result = validate_record(test_record, saved_reference)

print("Product code:", test_record["product_code"])
print("Validation result:", result)

assert result == "VALID"
print("Validator test passed!")

Product code: P101
Validation result: VALID
Validator test passed!


In [73]:
# Test different validation scenarios

test_records = [
    {
        "record_id": 501,
        "product_code": "P101",
        "event_date": "2026-05-15",
        "reference_version": 1
    },
    {
        "record_id": 502,
        "product_code": "P999",
        "event_date": "2026-05-15",
        "reference_version": 1
    },
    {
        "record_id": 503,
        "product_code": "P103",
        "event_date": "2026-08-15",
        "reference_version": 1
    },
    {
        "record_id": 504,
        "product_code": "P102",
        "event_date": "2026-08-15",
        "reference_version": 2
    }
]

for record in test_records:
    result = validate_record(record, saved_reference)
    print(record["product_code"], "->", result)

P101 -> VALID
P999 -> UNKNOWN_CODE
P103 -> EXPIRED_CODE
P102 -> DEPRECATED_CODE


In [74]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [75]:
!python -m pytest test_validator.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\Dell\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell\yottaflex-reference-pipeline
plugins: anyio-4.12.1
collecting ... collected 5 items

test_validator.py::test_valid_code PASSED                                [ 20%]
test_validator.py::test_unknown_code PASSED                              [ 40%]
test_validator.py::test_deprecated_code PASSED                           [ 60%]
test_validator.py::test_invalid_date PASSED                              [ 80%]
test_validator.py::test_expired_code PASSED                              [100%]

============================== 5 passed in 0.59s ==============================


In [76]:
!python -m pytest test_validator.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\Dell\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell\yottaflex-reference-pipeline
plugins: anyio-4.12.1
collecting ... collected 5 items

test_validator.py::test_valid_code PASSED                                [ 20%]
test_validator.py::test_unknown_code PASSED                              [ 40%]
test_validator.py::test_deprecated_code PASSED                           [ 60%]
test_validator.py::test_invalid_date PASSED                              [ 80%]
test_validator.py::test_expired_code PASSED                              [100%]

============================== 5 passed in 0.61s ==============================


In [77]:
!python -m pytest test_validator.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\Dell\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell\yottaflex-reference-pipeline
plugins: anyio-4.12.1
collecting ... collected 5 items

test_validator.py::test_valid_code PASSED                                [ 20%]
test_validator.py::test_unknown_code PASSED                              [ 40%]
test_validator.py::test_deprecated_code PASSED                           [ 60%]
test_validator.py::test_invalid_date PASSED                              [ 80%]
test_validator.py::test_expired_code PASSED                              [100%]

============================== 5 passed in 0.65s ==============================


In [78]:
import pipeline

print("Pipeline module imported successfully!")

Pipeline module imported successfully!


In [79]:
import pandas as pd

test_input = pd.DataFrame({
    "record_id": [701, 702, 703, 704],
    "product_code": ["P101", "P102", "P999", "P103"],
    "event_date": [
        "2026-05-15",
        "2026-08-15",
        "2026-08-15",
        "2026-08-15"
    ],
    "reference_version": [1, 2, 2, 1]
})

test_input.to_csv("test_input.csv", index=False)

print("Test input created successfully!")
display(test_input)

Test input created successfully!


,record_id,product_code,event_date,reference_version
0,701,P101,2026-05-15,1
1,702,P102,2026-08-15,2
2,703,P999,2026-08-15,2
3,704,P103,2026-08-15,1


In [80]:
import pipeline

result = pipeline.run_pipeline(
    input_file="test_input.csv",
    reference_file="versioned_reference.csv",
    output_dir="test_pipeline_runs"
)

print("\nPipeline test completed!")

PIPELINE COMPLETED
------------------
Run ID: 770bbfe58ff9fb99
Input records: 4
Accepted records: 1
Rejected records: 3
Reconciliation: PASSED
Execution time: 0.014 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\test_pipeline_runs\770bbfe58ff9fb99

Pipeline test completed!


In [81]:
import pandas as pd
from pathlib import Path

run_folder = Path("test_pipeline_runs") / result["run_id"]

rejected = pd.read_csv(run_folder / "rejected_records.csv")

display(
    rejected[
        ["record_id", "product_code",
         "validation_status", "rejection_reason"]
    ]
)

,record_id,product_code,validation_status,rejection_reason
0,702,P102,REJECTED,DEPRECATED_CODE
1,703,P999,REJECTED,UNKNOWN_CODE
2,704,P103,REJECTED,EXPIRED_CODE


In [82]:
first_run_id = result["run_id"]

second_result = pipeline.run_pipeline(
    input_file="test_input.csv",
    reference_file="versioned_reference.csv",
    output_dir="test_pipeline_runs"
)

second_run_id = second_result["run_id"]

print("First run ID:", first_run_id)
print("Second run ID:", second_run_id)

assert first_run_id == second_run_id

print("Repeatable run ID test PASSED!")

PIPELINE COMPLETED
------------------
Run ID: 770bbfe58ff9fb99
Input records: 4
Accepted records: 1
Rejected records: 3
Reconciliation: PASSED
Execution time: 0.017 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\test_pipeline_runs\770bbfe58ff9fb99
First run ID: 770bbfe58ff9fb99
Second run ID: 770bbfe58ff9fb99
Repeatable run ID test PASSED!


In [83]:
duplicate_input = pd.DataFrame({
    "record_id": [801, 801, 802],
    "product_code": ["P101", "P101", "P101"],
    "event_date": ["2026-05-15", "2026-05-15", "2026-05-15"],
    "reference_version": [1, 1, 1]
})

duplicate_input.to_csv("duplicate_test_input.csv", index=False)

duplicate_result = pipeline.run_pipeline(
    input_file="duplicate_test_input.csv",
    reference_file="versioned_reference.csv",
    output_dir="duplicate_test_runs"
)

print("Duplicate test completed!")

PIPELINE COMPLETED
------------------
Run ID: 7f20d7f5e70aa6be
Input records: 3
Accepted records: 2
Rejected records: 1
Reconciliation: PASSED
Execution time: 0.015 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\duplicate_test_runs\7f20d7f5e70aa6be
Duplicate test completed!


In [84]:
from pathlib import Path
import pandas as pd

duplicate_folder = Path("duplicate_test_runs") / duplicate_result["run_id"]

duplicate_rejections = pd.read_csv(
    duplicate_folder / "rejected_records.csv"
)

display(
    duplicate_rejections[
        ["record_id", "source_row", "rejection_reason"]
    ]
)

assert len(duplicate_rejections) == 1
assert duplicate_rejections.iloc[0]["rejection_reason"] == "DUPLICATE_RECORD_ID"

print("Duplicate rejection test PASSED!")

,record_id,source_row,rejection_reason
0,801,3,DUPLICATE_RECORD_ID


Duplicate rejection test PASSED!


In [121]:
missing_input = pd.DataFrame({
    "record_id": [901, 902, 903],
    "product_code": ["P101", None, "P101"],
    "event_date": ["2026-05-15", "2026-05-15", None],
    "reference_version": [1, 1, 1]
})

missing_input.to_csv("missing_test_input.csv", index=False)

missing_result = pipeline.run_pipeline(
    input_file="missing_test_input.csv",
    reference_file="versioned_reference.csv",
    output_dir="missing_test_runs"
)

print("Missing-value test completed!")

PIPELINE COMPLETED
------------------
Run ID: a1c8363b36eb9fc9
Input records: 3
Accepted records: 1
Rejected records: 2
Reconciliation: PASSED
Execution time: 0.017 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\missing_test_runs\a1c8363b36eb9fc9
Missing-value test completed!


In [111]:
from pathlib import Path
import pandas as pd

missing_folder = (
    Path("missing_test_runs") / missing_result["run_id"]
)

missing_rejections = pd.read_csv(
    missing_folder / "rejected_records.csv"
)

display(
    missing_rejections[
        ["record_id", "product_code", "rejection_reason"]
    ]
)

assert len(missing_rejections) == 2
assert missing_rejections["rejection_reason"].notna().all()

print("Missing-value rejection test PASSED!")

,record_id,product_code,rejection_reason
0,902,NaN,MISSING_PRODUCT_CODE
1,903,P101,MISSING_EVENT_DATE


Missing-value rejection test PASSED!


In [112]:
import pandas as pd
import time

small_benchmark = pd.concat(
    [test_input] * 250,
    ignore_index=True
)

small_benchmark["record_id"] = range(10001, 11001)

small_benchmark.to_csv(
    "benchmark_1000.csv",
    index=False
)

start = time.perf_counter()

benchmark_result = pipeline.run_pipeline(
    input_file="benchmark_1000.csv",
    reference_file="versioned_reference.csv",
    output_dir="benchmark_runs"
)

elapsed = time.perf_counter() - start

print("\nBENCHMARK RESULTS")
print("Records:", 1000)
print("Execution time:", round(elapsed, 3), "seconds")
print("Throughput:", round(1000 / elapsed, 2), "records/second")

PIPELINE COMPLETED
------------------
Run ID: ae89f9bb802b6c72
Input records: 1000
Accepted records: 250
Rejected records: 750
Reconciliation: PASSED
Execution time: 0.029 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\benchmark_runs\ae89f9bb802b6c72

BENCHMARK RESULTS
Records: 1000
Execution time: 0.05 seconds
Throughput: 19808.92 records/second


In [113]:
import importlib
import validator

importlib.reload(validator)

print("Updated validator loaded successfully!")

Updated validator loaded successfully!


In [114]:
import importlib
import pipeline

importlib.reload(pipeline)

print("Optimized pipeline loaded successfully!")

Optimized pipeline loaded successfully!


In [115]:
!python -m pytest test_validator.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\Dell\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell\yottaflex-reference-pipeline
plugins: anyio-4.12.1
collecting ... collected 5 items

test_validator.py::test_valid_code PASSED                                [ 20%]
test_validator.py::test_unknown_code PASSED                              [ 40%]
test_validator.py::test_deprecated_code PASSED                           [ 60%]
test_validator.py::test_invalid_date PASSED                              [ 80%]
test_validator.py::test_expired_code PASSED                              [100%]

============================== 5 passed in 0.57s ==============================


In [116]:
import time

start = time.perf_counter()

optimized_result = pipeline.run_pipeline(
    input_file="benchmark_1000.csv",
    reference_file="versioned_reference.csv",
    output_dir="optimized_benchmark_runs"
)

optimized_time = time.perf_counter() - start

print("\nOPTIMIZED BENCHMARK")
print("Records:", 1000)
print("Execution time:", round(optimized_time, 3), "seconds")
print("Throughput:", round(1000 / optimized_time, 2), "records/second")

PIPELINE COMPLETED
------------------
Run ID: ae89f9bb802b6c72
Input records: 1000
Accepted records: 250
Rejected records: 750
Reconciliation: PASSED
Execution time: 0.034 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\optimized_benchmark_runs\ae89f9bb802b6c72

OPTIMIZED BENCHMARK
Records: 1000
Execution time: 0.051 seconds
Throughput: 19651.89 records/second


In [117]:
import pandas as pd

# Create 100,000 records from the existing synthetic dataset
large_test_input = pd.concat(
    [test_input] * 25000,
    ignore_index=True
)

large_test_input["record_id"] = range(100001, 200001)

large_test_input.to_csv(
    "optimized_benchmark_100k.csv",
    index=False
)

print("100,000-record test dataset created!")
print("Total records:", len(large_test_input))

100,000-record test dataset created!
Total records: 100000


In [118]:
import pandas as pd

large_check = pd.read_csv("optimized_benchmark_100k.csv")

print("Total records:", len(large_check))
print("Columns:", list(large_check.columns))
print("Missing values:", large_check.isnull().sum().sum())
print("Unique record IDs:", large_check["record_id"].nunique())

assert len(large_check) == 100000
assert large_check["record_id"].nunique() == 100000

print("\n100K dataset verification PASSED!")

Total records: 100000
Columns: ['record_id', 'product_code', 'event_date', 'reference_version']
Missing values: 0
Unique record IDs: 100000

100K dataset verification PASSED!


In [119]:
import time

start = time.perf_counter()

large_result = pipeline.run_pipeline(
    input_file="optimized_benchmark_100k.csv",
    reference_file="versioned_reference.csv",
    output_dir="large_optimized_runs"
)

large_elapsed = time.perf_counter() - start

print("\n100K PERFORMANCE RESULTS")
print("Records processed:", large_result["input_count"])
print("Execution time:", round(large_elapsed, 2), "seconds")
print(
    "Throughput:",
    round(100000 / large_elapsed, 2),
    "records/second"
)

PIPELINE COMPLETED
------------------
Run ID: 4a289d05cc99779f
Input records: 100000
Accepted records: 25000
Rejected records: 75000
Reconciliation: PASSED
Execution time: 2.67 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\large_optimized_runs\4a289d05cc99779f

100K PERFORMANCE RESULTS
Records processed: 100000
Execution time: 3.57 seconds
Throughput: 28034.53 records/second


In [95]:
import importlib
import validator
import pipeline

importlib.reload(validator)
importlib.reload(pipeline)

print("Updated pipeline and validator loaded successfully!")

Updated pipeline and validator loaded successfully!


In [96]:
import importlib
import validator
import pipeline

importlib.reload(validator)
importlib.reload(pipeline)

print("Both modules reloaded successfully!")

Both modules reloaded successfully!


In [97]:
!python -m pytest test_validator.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\Dell\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell\yottaflex-reference-pipeline
plugins: anyio-4.12.1
collecting ... collected 5 items

test_validator.py::test_valid_code PASSED                                [ 20%]
test_validator.py::test_unknown_code PASSED                              [ 40%]
test_validator.py::test_deprecated_code PASSED                           [ 60%]
test_validator.py::test_invalid_date PASSED                              [ 80%]
test_validator.py::test_expired_code PASSED                              [100%]

============================== 5 passed in 0.72s ==============================


In [98]:
import importlib
import pipeline

importlib.reload(pipeline)

final_test = pipeline.run_pipeline(
    input_file="test_input.csv",
    reference_file="versioned_reference.csv",
    output_dir="final_test_runs"
)

assert final_test["input_count"] == 4
assert final_test["accepted_count"] == 1
assert final_test["rejected_count"] == 3
assert final_test["reconciliation_passed"] is True

print("\nEND-TO-END PIPELINE TEST PASSED!")

PIPELINE COMPLETED
------------------
Run ID: 770bbfe58ff9fb99
Input records: 4
Accepted records: 1
Rejected records: 3
Reconciliation: PASSED
Execution time: 0.015 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\final_test_runs\770bbfe58ff9fb99

END-TO-END PIPELINE TEST PASSED!


In [99]:
from pathlib import Path
import pandas as pd

large_folder = (
    Path("large_optimized_runs") / large_result["run_id"]
)

large_output = pd.read_csv(
    large_folder / "all_results.csv"
)

print("Output records:", len(large_output))
print("Missing validation statuses:",
      large_output["validation_status"].isna().sum())

print("\nValidation status counts:")
display(large_output["rejection_reason"].value_counts(dropna=False))

assert len(large_output) == 100000
assert large_output["validation_status"].notna().all()

print("100K output verification PASSED!")

Output records: 100000
Missing validation statuses: 0

Validation status counts:


rejection_reason
NaN                25000
DEPRECATED_CODE    25000
UNKNOWN_CODE       25000
EXPIRED_CODE       25000
Name: count, dtype: int64

100K output verification PASSED!


In [100]:
print("FINAL VALIDATION STATUS COUNTS")

display(
    large_output["validation_status"].value_counts()
)

accepted = (
    large_output["validation_status"] == "VALID"
).sum()

rejected = (
    large_output["validation_status"] == "REJECTED"
).sum()

print("Accepted:", accepted)
print("Rejected:", rejected)
print("Total:", accepted + rejected)

assert accepted == 25000
assert rejected == 75000
assert accepted + rejected == 100000

print("\nFINAL RECONCILIATION PASSED!")

FINAL VALIDATION STATUS COUNTS


validation_status
REJECTED    75000
VALID       25000
Name: count, dtype: int64

Accepted: 25000
Rejected: 75000
Total: 100000

FINAL RECONCILIATION PASSED!


In [101]:
import importlib
import pipeline

importlib.reload(pipeline)

print("Safe-publishing pipeline loaded successfully!")

Safe-publishing pipeline loaded successfully!


In [102]:
!python -m pytest test_validator.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\Dell\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell\yottaflex-reference-pipeline
plugins: anyio-4.12.1
collecting ... collected 5 items

test_validator.py::test_valid_code PASSED                                [ 20%]
test_validator.py::test_unknown_code PASSED                              [ 40%]
test_validator.py::test_deprecated_code PASSED                           [ 60%]
test_validator.py::test_invalid_date PASSED                              [ 80%]
test_validator.py::test_expired_code PASSED                              [100%]

============================== 5 passed in 0.66s ==============================


In [103]:
import importlib
import pipeline

importlib.reload(pipeline)

safe_test = pipeline.run_pipeline(
    input_file="test_input.csv",
    reference_file="versioned_reference.csv",
    output_dir="safe_test_runs"
)

assert safe_test["input_count"] == 4
assert safe_test["accepted_count"] == 1
assert safe_test["rejected_count"] == 3
assert safe_test["reconciliation_passed"]

print("SAFE-PUBLISHING TEST PASSED!")

PIPELINE COMPLETED
------------------
Run ID: 770bbfe58ff9fb99
Input records: 4
Accepted records: 1
Rejected records: 3
Reconciliation: PASSED
Execution time: 0.019 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\safe_test_runs\770bbfe58ff9fb99
SAFE-PUBLISHING TEST PASSED!


In [107]:
second_safe_test = pipeline.run_pipeline(
    input_file="test_input.csv",
    reference_file="versioned_reference.csv",
    output_dir="safe_test_runs"
)

assert second_safe_test["run_id"] == safe_test["run_id"]
assert second_safe_test["input_count"] == 4
assert second_safe_test["accepted_count"] == 1
assert second_safe_test["rejected_count"] == 3

from pathlib import Path

folder = Path("safe_test_runs") / second_safe_test["run_id"]

print("\nOUTPUT FILES:")
for file in sorted(folder.iterdir()):
    print(file.name)

assert not list(folder.glob("*.tmp"))

print("\nSAFE RERUN TEST PASSED!")

PIPELINE COMPLETED
------------------
Run ID: 770bbfe58ff9fb99
Input records: 4
Accepted records: 1
Rejected records: 3
Reconciliation: PASSED
Execution time: 0.014 seconds
Output folder: C:\Users\Dell\yottaflex-reference-pipeline\safe_test_runs\770bbfe58ff9fb99

OUTPUT FILES:
all_results.csv
rejected_records.csv
rejection_summary.csv
run_log.json
valid_records.csv

SAFE RERUN TEST PASSED!


In [105]:
!python -m pytest test_pipeline.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\Dell\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell\yottaflex-reference-pipeline
plugins: anyio-4.12.1
collecting ... collected 5 items

test_pipeline.py::test_pipeline_counts_and_reconciliation PASSED         [ 20%]
test_pipeline.py::test_pipeline_creates_output_files PASSED              [ 40%]
test_pipeline.py::test_pipeline_rejection_reasons PASSED                 [ 60%]
test_pipeline.py::test_pipeline_repeatable_run_id PASSED                 [ 80%]
test_pipeline.py::test_pipeline_run_log PASSED                           [100%]

============================== 5 passed in 0.81s ==============================


In [106]:
!python -m pytest -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- C:\Users\Dell\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell\yottaflex-reference-pipeline
plugins: anyio-4.12.1
collecting ... collected 10 items

test_pipeline.py::test_pipeline_counts_and_reconciliation PASSED         [ 10%]
test_pipeline.py::test_pipeline_creates_output_files PASSED              [ 20%]
test_pipeline.py::test_pipeline_rejection_reasons PASSED                 [ 30%]
test_pipeline.py::test_pipeline_repeatable_run_id PASSED                 [ 40%]
test_pipeline.py::test_pipeline_run_log PASSED                           [ 50%]
test_validator.py::test_valid_code PASSED                                [ 60%]
test_validator.py::test_unknown_code PASSED                              [ 70%]
test_validator.py::test_deprecated_code PASSED                           [ 80%]
test_validator.py::test_invalid_date PASSED      